In [1]:
import pandas as pd
import os

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import TavilySearchTool, ScrapeWebsiteTool
from pydantic import BaseModel, Field
from typing import List, Optional

In [2]:
from dotenv import load_dotenv
load_dotenv("../.env.local", override=True)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_KEY = os.getenv("OPEN_AI_API_KEY")

In [3]:
search_tool = TavilySearchTool(api_key=TAVILY_API_KEY)
scrape_tool = ScrapeWebsiteTool()  
llm = LLM(model="gpt-4o", api_key=OPENAI_API_KEY)

In [49]:
class ReleaseCandidate(BaseModel):
    product_name: str = Field(..., description="Name of the product")
    brand: Optional[str] = Field(None, description="Brand if known")
    release_date: str = Field(..., description="Date product releases")
    retail_price: Optional[int] = Field(..., description="Expected retail price of product")
    retailers: Optional[str] = Field(..., description="Retailers confirmed to be selling the item")
    seed_sources: List[str] = Field(..., description="URLs confirming the release")

class ScoutOutput(BaseModel):
    candidates: List[ReleaseCandidate]

class ReleaseItems(ReleaseCandidate):
    resale_estimate: int = Field(..., description="Estimated resale value")
    confidence_score: float = Field(..., description="Level of confidence from 0-100 that resale_estimate is correct")

class AnalystOutput(BaseModel):
    items: List[ReleaseItems]

## Agents

In [50]:
sneaker_scout = Agent(
    role="Upcoming Sneaker Release Scout",
    goal="""Identify upcoming sneaker releases.""",
    backstory="""
    You are a master web scraper who is highly resourceful and can easily navigate the internet to find relevant information and 
    Extract it in easily ingestible formats. You do not get distracted by irrelevant articles/information.
    """,
    tools=[search_tool, scrape_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

sneaker_market_analyst = Agent(
    role="Sneaker Resell Market Analyst",
    goal="Accurately project resale values for upcoming sneaker releases",
    backstory="""You are a long-time sneaker reseller and hypebeast. You have an expert understanding of how cultural trends,
    historical performance and market factors impact the potential profitability of sneakers on the secondary market.""",
    tools=[search_tool, scrape_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)


## Tasks

In [53]:
sneaker_scout_task = Task(
    description="""
    Compile information on upcoming sneaker releases by scraping information from reputable release calendars on
    sites such as https://www.sneakerfiles.com/release-dates/, nicekicks.com, sneakernews.com, and goat.com. Find and navigate to the release calendar pages on each site
    to find organized information on upcoming releases.
    
    - Today is {today}. Only include releases between {today} and {cutoff_date}.

    Deliverable: Return up to {num_items} releases ordered by the soonest upcoming release.
    """,
    expected_output="""ScoutOutput with exactly {num_items} upcoming sneaker releases in the date window, each with 1–2 sources. You're outputted
    items should closely match the items on the release calendars of the sites.""",
    output_pydantic=ScoutOutput,
    agent=sneaker_scout,
)

sneaker_market_analyst_task = Task(
    description="""
    Given a list of upcoming sneaker releases, do research and analyze each item one by one to come up with a resell price prediction
    and confidence score for that prediction.
    Consider historical trends and the performance of similar sneakers (same model or release type (collaboration, limited release etc.)
    Use StockX sale price as the most reliable indicator for fair resale value.
    Based on your research, make a resell price prediction. If you are considering a wide range of values, choose the 50th percentile value
    and lower your confidence score.
    If you can't find similar items, or there is no history of consistent price trends for similar items, lower your confidence score
    significantly.

    Confidence Score Rubric:
    75-100: Highly confident that the show will consistently resell within $20 of the resale prediction. There is a clear trend of 
    very similar items reselling at this price, and there are little to no factors that could cause this item not to follow that trend
    and closely match the prediction.
    50-75: Moderately confident that the resale prediction is accurate, but potential outside factors could cause the prediction to 
    be incorrect.
    25-50: You are mostly unsure about this prediction. Price trends for similar items are inconsistent, or the item is unique and
    hard to compare to other releases. 
    0-25: This is a truly unique release and incomparable to anything else. You have very little confidence in your resale estimate.

    Important note: Using StockX sale value for the exact item you are analyzing is not reliable because the item has not released to 
    the public yet and pre-release prices are always inflated.
    Instead, look at prices for previously released items of the same model or line as the sneaker you are analyzing. Use your intuition
    about what makes items popular, to evaluate unique items such as collaborations, and limited releases.

    Deliverable: For each item in the given list, generate a resale price prediction of what you think the item will sell for on the
    secondary resell market within one month of purchase and a confidence score based on how confident you are in that prediction.
    You will output AnalystOutput with predictions and confidence scores for all items provided in the original list.
    """,
    expected_output="""
    AnalystOutput with the same number of items as the original given list, including resale price predictions and confidence scores
    for each item.
    """,
    output_pydantic=AnalystOutput,
    agent=sneaker_market_analyst,
    context=[sneaker_scout_task],
)

In [54]:
from datetime import date, timedelta

today = date.today()
cutoff = today + timedelta(days=21)
window_month = date.today()

# Change number of items
num_items = 3 

test_crew = Crew(
    agents=[sneaker_scout, sneaker_market_analyst],
    tasks=[sneaker_scout_task, sneaker_market_analyst_task],
    verbose=True,
    process=Process.sequential,
)

result = test_crew.kickoff(
    inputs={
        "today": today.isoformat(),
        "cutoff_date": cutoff.isoformat(),
        "num_items": num_items
    }
)

result

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9a6a7e28-c7f9-4afa-8cab-588ad7265466                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/caedinm/Projects/flipper/venv/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Upcoming Sneaker Release Scout                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Compile information on upcoming sneaker releases by scraping information from reputable release calendars  │
│  on                                                                                                             │
│      sites such as https://www.sneakerfiles.com/release-dates/, nicekicks.com, sneakernews.com, and goat.com.   │
│  Find and navigate to the release calendar pages on each site                                                   │
│      to find organized information on upcoming releases.                                                        │
│                                                                                                                 │
│      - Today is 2025-12-30. Only include releases between 2025-12-30 and 2026-01-20.                            │
│                                                                                                                 │
│      Deliverable: Return up to 3 releases ordered by the soonest upcoming release.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/Users/caedinm/Projects/flipper/venv/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Upcoming Sneaker Release Scout                                                                          │
│                                                                                                                 │
│  Thought: Thought: I will first search for the release calendar page URLs on the specified sites to gather      │
│  information about upcoming sneaker releases.                                                                   │
│                                                                                                                 │
│  Using Tool: Tavily Search                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sneaker Resell Market Analyst                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "items": [                                                                                                   │
│      {                                                                                                          │
│        "product_name": "Nike Book 1",                                                                           │
│        "brand": "Nike",                                                                                         │
│        "release_date": "2025-12-30",                                                                            │
│        "retail_price": 155,                                                                                     │
│        "retailers": "Nikestore US",                                                                             │
│        "seed_sources": [                                                                                        │
│          "https://www.sneakerfiles.com/release-dates/",                                                         │
│          "https://sneakernews.com/release-dates/"                                                               │
│        ],                                                                                                       │
│        "resale_estimate": 195,                                                                                  │
│        "confidence_score": 75                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "product_name": "Nike A'One",                                                                            │
│        "brand": "Nike",                                                                                         │
│        "release_date": "2026-01-01",                                                                            │
│        "retail_price": 115,                                                                                     │
│        "retailers": "Nikestore US",                                                                             │
│        "seed_sources": [                                                                                        │
│          "https://sneakernews.com/release-dates/",                                                              │
│          "https://www.nicekicks.com/sneaker-release-dates/"                                                     │
│        ],                                                                                                       │
│        "resale_estimate": 250,                                                                                  │
│        "confidence_score": 70                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "product_name": "Nike Kobe 9 EM",               

CrewOutput(raw='{\n  "items": [\n    {\n      "product_name": "Nike Book 1",\n      "brand": "Nike",\n      "release_date": "2025-12-30",\n      "retail_price": 155,\n      "retailers": "Nikestore US",\n      "seed_sources": [\n        "https://www.sneakerfiles.com/release-dates/",\n        "https://sneakernews.com/release-dates/"\n      ],\n      "resale_estimate": 195,\n      "confidence_score": 75\n    },\n    {\n      "product_name": "Nike A\'One",\n      "brand": "Nike",\n      "release_date": "2026-01-01",\n      "retail_price": 115,\n      "retailers": "Nikestore US",\n      "seed_sources": [\n        "https://sneakernews.com/release-dates/",\n        "https://www.nicekicks.com/sneaker-release-dates/"\n      ],\n      "resale_estimate": 250,\n      "confidence_score": 70\n    },\n    {\n      "product_name": "Nike Kobe 9 EM",\n      "brand": "Nike",\n      "release_date": "2026-01-01",\n      "retail_price": 190,\n      "retailers": "Nike SNKRS US, DICK\'S",\n      "seed_sources

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 890502dc-812a-49e0-aba2-3af70ec9f7c6                                                                     │
│  Agent: Sneaker Resell Market Analyst                                                                           │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9a6a7e28-c7f9-4afa-8cab-588ad7265466                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "items": [                                                                                                   │
│      {                                                                                                          │
│        "product_name": "Nike Book 1",                                                                           │
│        "brand": "Nike",                                                                                         │
│        "release_date": "2025-12-30",                                                                            │
│        "retail_price": 155,                                                                                     │
│        "retailers": "Nikestore US",                                                                             │
│        "seed_sources": [                                                                                        │
│          "https://www.sneakerfiles.com/release-dates/",                                                         │
│          "https://sneakernews.com/release-dates/"                                                               │
│        ],                                                                                                       │
│        "resale_estimate": 195,                                                                                  │
│        "confidence_score": 75                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "product_name": "Nike A'One",                                                                            │
│        "brand": "Nike",                                                                                         │
│        "release_date": "2026-01-01",                                                                            │
│        "retail_price": 115,                                                                                     │
│        "retailers": "Nikestore US",                                                                             │
│        "seed_sources": [                                                                                        │
│          "https://sneakernews.com/release-dates/",                                                              │
│          "https://www.nicekicks.com/sneaker-release-dates/"                                                     │
│        ],                                                                                                       │
│        "resale_estimate": 250,                                                                                  │
│        "confidence_score": 70                                                                                   │
│      },                                                                                                         │
│      {                                                

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯